# 🔎 Semantic Search with Transformers

In [ ]:
# Run this cell beforehand so you do not see any warnings
import warnings
warnings.filterwarnings('ignore')

In [ ]:
import os, subprocess

print("=== Environment ===")
print(f"LD_LIBRARY_PATH: {os.environ.get('LD_LIBRARY_PATH', 'NOT SET')}")
print(f"CUDA_VISIBLE_DEVICES: {os.environ.get('CUDA_VISIBLE_DEVICES', 'NOT SET')}")
print(f"PATH: {os.environ.get('PATH', '')[:200]}")

print("\n=== nvidia-smi ===")
print(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout or "FAILED")

print("\n=== libcuda.so ===")
print(subprocess.run(["find", "/usr/lib64", "/usr/lib", "/usr/local", "-name", "libcuda.so*", "-maxdepth", "3"], capture_output=True, text=True).stdout or "NOT FOUND")

print("\n=== PyTorch ===")
import torch
print(f"torch.version.cuda: {torch.version.cuda}")
print(f"torch.cuda.is_available(): {torch.cuda.is_available()}")
print(f"torch.backends.cudnn.enabled: {torch.backends.cudnn.enabled}")


## Import the Libraries

In [ ]:
import pickle
import pandas as pd
import torch
import numpy as np
import faiss
import os

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn import preprocessing

## Load the Data

In [ ]:
df = pd.read_json("data/research_papers.json")

In [ ]:
df = df.drop(["author", "link", "tag"], axis=1)
df.head()

In [ ]:
print(f"Number of research papers: {len(df)}")
pd.set_option('display.max_colwidth', None)
df.head()

## Retrieve the Model

In [ ]:
# model source: @ https://huggingface.co/sentence-transformers/models
model = SentenceTransformer('all-mpnet-base-v2')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
print(f"model: {model}\n device: {device}")

## Generate or Load the Embeddings

In [ ]:
# One-Off: Generate embeddings on a GPU-enabled environment preferably
embeddings = model.encode(df.summary.to_list(), show_progress_bar=True)
with open('data/new_embeddings.pickle', 'wb') as pkl:
    pickle.dump(embeddings, pkl)

In [ ]:
# Util method to load embeddings
def load_embeddings(file_path:str, mode: str = 'rb'):
    with open(file_path, mode) as f:
        embeddings = pickle.load(f)
        return embeddings, len(embeddings), embeddings.shape

In [ ]:
embeddings, length, shape = load_embeddings('data/new_embeddings.pickle')
print(f"embeddings: {embeddings[0]}\n length: {length}\n shape: {shape}")
print(f"Is instance of numpy arrays :{isinstance(embeddings, np.ndarray)}")

In [ ]:
print(f"first paper: {embeddings[0].shape}")

## Data Preparation and Helper Methods

In [ ]:
label_encoder = preprocessing.LabelEncoder()
print(f"Data type before encoding: {df['id'].dtype}")
df.head()


In [ ]:
df['encoded_id'] = label_encoder.fit_transform(df['id'])
print(f"Data type after encoding: {df['encoded_id'].dtype}")
df.head()

In [ ]:
"""
Method to return a list of column values for papers specified by their IDs
parameters:
  df: The DataFrame in which the data is contained
  I: List of IDs of the papers for which the information is required
  column: Column of the DataFrame where the required information is stored
"""
def id_to_info(df, I, column):
    print(f"df: {df}\n I: {I}\n column: {column}")
    return [list(df[df['encoded_id'] == idx][column]) for idx in I]

## Set up the Index

In [ ]:
embeddings_np = np.array(embeddings, dtype=np.float32)
print(f"Numpy embeddings: {embeddings_np}\n data type: {embeddings.dtype}")

In [ ]:
def create_gpu_index(embedding_dim):
    num_gpus = faiss.get_num_gpus()
    print(f"Faiss detected {num_gpus} GPU(s)")

    if num_gpus == 0:
        print("No GPU available for Faiss, falling back to CPU index")
        index = faiss.IndexFlatL2(embedding_dim)
        return faiss.IndexIDMap(index)

    # Always use device 0 — SLURM's CUDA_VISIBLE_DEVICES already remaps
    # physical GPUs so the app always sees device 0
    res = faiss.StandardGpuResources()
    config = faiss.GpuIndexFlatConfig()
    config.device = 0

    gpu_index = faiss.GpuIndexFlatL2(res, embedding_dim, config)
    return faiss.IndexIDMap(gpu_index)

In [ ]:
gpu_index_map = create_gpu_index(embeddings_np.shape[1])

In [ ]:
gpu_index_map.add_with_ids(
    embeddings_np, df["encoded_id"][:length].values.astype("int64")
)

print(f"Number of embeddings in the Faiss index: {gpu_index_map.ntotal}")

In [ ]:
## Search with a Summary
df.iloc[1337, [3, 1]]

In [ ]:
# Search by existing paper summary
D, I = gpu_index_map.search(np.array([embeddings[1337]]), k=10)
pd.DataFrame({'L2 distance': D.flatten().tolist(), 'ML paper IDs': I.flatten().tolist(), 'ML paper titles': id_to_info(df, I.flatten(), 'title'), 'Summaries': id_to_info(df, I.flatten(), 'summary')}).head(10)

## Prompt Search


In [ ]:
query = "The dominant sequence transduction models are based on complex recurrent or convolutional neural networks in an encoder-decoder configuration. The best performing models also connect the encoder and decoder through an attention mechanism. We propose a new simple network architecture, the Transformer, based solely on attention mechanisms, dispensing with recurrence and convolutions entirely. Experiments on two machine translation tasks show these models to be superior in quality while being more parallelizable and requiring significantly less time to train. Our model achieves 28.4 BLEU on the WMT 2014 English-to-German translation task, improving over the existing best results, including ensembles by over 2 BLEU. On the WMT 2014 English-to-French translation task, our model establishes a new single-model state-of-the-art BLEU score of 41.8 after training for 3.5 days on eight GPUs, a small fraction of the training costs of the best models from the literature. We show that the Transformer generalizes well to other tasks by applying it successfully to English constituency parsing both with large and limited training data"

In [ ]:
embed = model.encode([query])

print(f"embed shape: {embed.shape}")
print(f"embed type: {type(embed)}")

In [ ]:

D, I = gpu_index_map.search(embed.astype("float32"), k=10)

results = {'L2 distances':D.flatten().tolist(), 'ML paper IDs':I.flatten().tolist(), "Titles": id_to_info(df, I.flatten(), 'title'), "Summaries": id_to_info(df, I.flatten(), 'summary')}

pd.DataFrame(results).head(10)